# データ保存

In [ ]:
import os
os.environ["SPEDAS_DATA_DIR"] = "/mnt/j/observation_data/"

# 必要なデータ
- 軌道データ (SM座標系)
- 電子 (omni flux, PA) (~10 keV)
- proton (omni flux, PA) (~ 1 keV)
- 数密度 (Arase: LEP-e & HFA, THEMIS: mom)
- 温度 (Arase: ion平均と電子, THEMIS: proton想定)
- $V_{\mathrm{sys}\perp}$ (Arase: ion平均考慮)
- $B_{\mathrm{tot}}$
- $B_{x}$
- $B_{y}$
- $B_{z}$
- $S_{z}, \bf{S}_{\perp}$

# 軌道データの作成

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

pt.del_data('*')

trange = ['2022-09-01/22:25', '2022-09-01/23:15']

earth_radius = 6378.1  # km

psp.themis.state(trange=trange, probe='a', no_update=True)
psp.cotrans(name_in='tha_pos_gse', name_out='tha_pos_sm', coord_in='gse', coord_out='sm')   # 'tha_pos_sm'

themis_a_pos_sm = pt.data_quants['tha_pos_sm']
themis_a_pos_sm.values = themis_a_pos_sm.values / earth_radius  # convert to RE
themis_a_pos_sm.attrs['Units'] = 'R_E'
# trangeに合わせてデータを切り出し
themis_a_pos_sm = themis_a_pos_sm.sel(time=slice(trange[0], trange[1]))

print(themis_a_pos_sm)

psp.erg.orb(trange=trange, level='l2', datatype='def', no_update=True)  # 'erg_orb_l2_pos_sm'
Arase_pos_sm = pt.data_quants['erg_orb_l2_pos_sm']
Arase_pos_sm = Arase_pos_sm.sel(time=slice(trange[0], trange[1]))

print(Arase_pos_sm)

In [ ]:
Arase_pos_sm_x, Arase_pos_sm_y, Arase_pos_sm_z = Arase_pos_sm.values[:,0], Arase_pos_sm.values[:,1], Arase_pos_sm.values[:,2]
Arase_pos_sm_time = Arase_pos_sm.time
themis_a_pos_sm_x, themis_a_pos_sm_y, themis_a_pos_sm_z = themis_a_pos_sm.values[:,0], themis_a_pos_sm.values[:,1], themis_a_pos_sm.values[:,2]
themis_a_pos_sm_time = themis_a_pos_sm.time

In [ ]:
# 文字の大きさ
mpl.rcParams['font.size'] = 25

In [ ]:
def pretty(ax, xlabel, ylabel):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.minorticks_on()
    ax.grid(which='both', linestyle='--', alpha=0.7)
    ax.axhline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.axvline(0, color='k', lw=1, linestyle='-.', alpha=0.7)
    ax.set_aspect('equal', 'box')

fig = plt.figure(figsize=(18, 6), dpi=300)

ax1 = fig.add_subplot(131)
ax1.plot(themis_a_pos_sm_x, themis_a_pos_sm_y, color='magenta', label='THEMIS-A', linewidth=3)
ax1.scatter(themis_a_pos_sm_x[0],  themis_a_pos_sm_y[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax1.scatter(themis_a_pos_sm_x[-1], themis_a_pos_sm_y[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax1.plot(Arase_pos_sm_x, Arase_pos_sm_y, color='green', label='Arase', linewidth=3)
ax1.scatter(Arase_pos_sm_x[0],  Arase_pos_sm_y[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax1.scatter(Arase_pos_sm_x[-1], Arase_pos_sm_y[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax1, 'X‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Y‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax1.set_xlim(1, -9)
ax1.set_ylim(8.5, -1.5)


ax2 = fig.add_subplot(132)
ax2.plot(themis_a_pos_sm_x, themis_a_pos_sm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax2.scatter(themis_a_pos_sm_x[0],  themis_a_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax2.scatter(themis_a_pos_sm_x[-1], themis_a_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax2.plot(Arase_pos_sm_x, Arase_pos_sm_z, color='green', label='Arase', linewidth=3)
ax2.scatter(Arase_pos_sm_x[0],  Arase_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax2.scatter(Arase_pos_sm_x[-1], Arase_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax2, 'X‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Z‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax2.set_xlim(1, -9)
ax2.set_ylim(-3, 7)

ax3 = fig.add_subplot(133)
ax3.plot(themis_a_pos_sm_y, themis_a_pos_sm_z, color='magenta', label='THEMIS-A', linewidth=3)
ax3.scatter(themis_a_pos_sm_y[0],  themis_a_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='magenta', linewidths=1)
ax3.scatter(themis_a_pos_sm_y[-1], themis_a_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='magenta', linewidths=1)
ax3.plot(Arase_pos_sm_y, Arase_pos_sm_z, color='green', label='Arase', linewidth=3)
ax3.scatter(Arase_pos_sm_y[0],  Arase_pos_sm_z[0],  marker='o', c='yellow', s=200, edgecolors='green', linewidths=1)
ax3.scatter(Arase_pos_sm_y[-1], Arase_pos_sm_z[-1], marker='*', c='yellow', s=400, edgecolors='green', linewidths=1)
pretty(ax3, 'Y‒SM\n'+ r'[$R_{\mathrm{E}}$]', 'Z‒SM\n'+ r'[$R_{\mathrm{E}}$]')
ax3.set_xlim(5, -1)
ax3.set_ylim(-1, 5)

for ax in (ax1, ax2, ax3):
    th = np.linspace(0, 2*np.pi, 256)
    ax.plot(np.cos(th), np.sin(th), 'k-', lw=1, alpha=1)
    if ax == ax1 or ax == ax2:
        # 円の中のx<0の部分を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
    if ax == ax3:
        # 円の中を塗りつぶす
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)<0), color='k')
        ax.fill_betweenx(np.sin(th), np.cos(th), 0, where=(np.cos(th)>0), color='k')
    
    # 各図の外左上に(a-1) (a-2) (a-3) の文字を入れる
    if ax == ax1:
        ax.text(-0.25, 0.95, '(a-1)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax2:
        ax.text(-0.25, 0.95, '(a-2)', transform=ax.transAxes, verticalalignment='top')
    elif ax == ax3:
        ax.text(-0.25, 0.95, '(a-3)', transform=ax.transAxes, verticalalignment='top')

fig.tight_layout()
plt.show()

fig.savefig(r"/mnt/j/KAW_observation/Figure_1_a.pdf")

# 電子・proton omniflux, PA distribution

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import xarray as xr

pt.del_data('*')

trange = ['2022-09-01/22:25', '2022-09-01/23:15']

psp.erg.lepe(trange=trange, level='l2', datatype='3dflux', no_update=True)
psp.erg.lepe(trange=trange, level='l2', datatype='omniflux', no_update=True)
psp.erg.lepi(trange=trange, level='l2', datatype='3dflux', no_update=True)
psp.erg.lepi(trange=trange, level='l2', datatype='omniflux', no_update=True)
psp.erg.mgf(trange=trange, level='l2', datatype='64hz', coord='dsi', no_update=True)
psp.erg.orb(trange=trange, level='l2', no_update=True)

In [ ]:
psp.projects.erg.erg_lep_part_products(
    'erg_lepe_l2_3dflux_FEDU',
    outputs=['pa'],
    pitch=[0, 180],
    energy=[40, 5000],
    mag_name='erg_mgf_l2_mag_64hz_dsi',
    pos_name='erg_orb_l2_pos_sm'
)

psp.projects.erg.erg_lep_part_products(
    'erg_lepi_l2_3dflux_FPDU',
    outputs=['pa'],
    pitch=[0, 180],
    energy=[40, 1000],
    mag_name='erg_mgf_l2_mag_64hz_dsi',
    pos_name='erg_orb_l2_pos_sm'
)

In [ ]:
mpl.rcParams['font.size'] = 20

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import pytplot as pt

def omniflux_plot_Arase(inst_name, fig, ax, cax, dq, vmin=None, vmax=None, energy_min=None, energy_max=None, ytitle=None, ztitle=None):
    # 取り出し（全て (T, E) 形状）
    flux = np.asarray(dq.data, dtype=float)           # (T, E)
    if inst_name == 'lepe':
        E    = np.asarray(dq.spec_bins, dtype=float)      # (T, E) 変動エネルギー
    elif inst_name == 'lepi':
        E    = np.asarray(dq.spec_bins, dtype=float)      # (E) 変動エネルギー
        E    = E * 1E3                                     # keV → eV
        E    = np.broadcast_to(E[None, :], flux.shape)      # (T, E)
    t    = np.asarray(dq.time.values)                 # (T,)

    # pcolormeshのX/Yは非有限NG → 列方向で全部有限なチャンネルだけ残す
    good_e = np.all(~np.isnan(E), axis=0)
    if not np.all(good_e):
        flux = flux[:, good_e]
        E    = E[:,    good_e]

    # ---- 値の前処理（LogNorm 用）----
    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan
    if np.all(~np.isfinite(flux)):
        raise ValueError("有効な（>0）フラックスがありません。")

    if vmin == None:
        vmin = np.nanmin(flux)
    if vmax == None:
        vmax = np.nanmax(flux)
    if not (vmin < vmax):
        vmax = vmin * 1.0001
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    print(vmin, vmax)

    flux[np.isnan(flux)] = 1E-99

    # ---- 時間メッシュ生成（(T,E)）----
    Tmesh = np.broadcast_to(t[:, None], E.shape)

    print(flux.shape)
    print(E.shape)
    print(Tmesh.shape)

    # ---- 描画 ----
    mesh = ax.pcolormesh(Tmesh, E, flux, norm=norm, cmap=cm.turbo, shading='nearest')
    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label("")  # ラベル消す
    #if ztitle:
    #    ax.text(0.85, 0.15, ztitle,
    #            transform=ax.transAxes, ha='center', va='center', color='k',
    #            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.25'))

    ax.set_yscale('log')
    ax.set_ylabel(ytitle)
    ax.grid(which='both', alpha=0.5)
    ax.minorticks_on()

    if energy_min != None:
        ax.set_ylim(ymin=energy_min)
    if energy_max != None:
        ax.set_ylim(ymax=energy_max)

    # 時刻目盛り
    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    return fig, ax, cax

def pa_plot_Arase(fig, ax, cax, dq, vmin=None, vmax=None, ytitle=None, ztitle=None):
    # 取り出し（全て (T, E) 形状）
    flux = np.asarray(dq.data, dtype=float)           # (T, E)
    PA    = np.asarray(dq.spec_bins, dtype=float)     # (T, E)
    t    = np.asarray(dq.time.values)                 # (T,)

    # pcolormeshのX/Yは非有限NG → 列方向で全部有限なチャンネルだけ残す
    good_PA = np.all(~np.isnan(PA), axis=0)
    if not np.all(good_PA):
        flux = flux[:, good_PA]
        PA    = PA[:,    good_PA]

    # ---- 値の前処理（LogNorm 用）----
    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan
    if np.all(~np.isfinite(flux)):
        raise ValueError("有効な（>0）フラックスがありません。")

    if vmin == None:
        vmin = np.nanmin(flux)
    if vmax == None:
        vmax = np.nanmax(flux)
    if not (vmin < vmax):
        vmax = vmin * 1.0001
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    print(vmin, vmax)

    flux[np.isnan(flux)] = 1E-99

    # ---- 時間メッシュ生成（(T,E)）----
    Tmesh = np.broadcast_to(t[:, None], PA.shape)

    print(flux.shape)
    print(PA.shape)
    print(Tmesh.shape)

    # ---- 描画 ----
    mesh = ax.pcolormesh(Tmesh, PA, flux, norm=norm, cmap=cm.turbo, shading='nearest')
    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label("")  # ラベル消す
    #if ztitle:
    #    ax.text(0.85, 0.5, ztitle,
    #            transform=ax.transAxes, ha='center', va='center', color='k',
    #            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.25'))

    ax.set_ylabel(ytitle)
    ax.grid(which='both', alpha=0.5)
    ax.minorticks_on()

    ax.set_ylim(ymax=180, ymin=0)
    ax.set_yticks(np.arange(0, 181, 45))

    # 時刻目盛り
    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    return fig, ax, cax

def add_panel_label(ax, label, x=-0.08, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

fig = plt.figure(figsize=(10, 12))
gs = fig.add_gridspec(4, 2, width_ratios=[1, 0.025], wspace=0.05)
ax_1 = fig.add_subplot(gs[0, 0])
cax_1 = fig.add_subplot(gs[0, 1])
ax_2 = fig.add_subplot(gs[1, 0], sharex=ax_1)
cax_2 = fig.add_subplot(gs[1, 1])
ax_3 = fig.add_subplot(gs[2, 0], sharex=ax_1)
cax_3 = fig.add_subplot(gs[2, 1])
ax_4 = fig.add_subplot(gs[3, 0], sharex=ax_1)
cax_4 = fig.add_subplot(gs[3, 1])

ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)

dq_proton = pt.data_quants['erg_lepi_l2_omniflux_FPDO'].sel(time=slice(*trange)) * 1E-3 # '#/s/$cm^{2}$/str/eV'
dq_proton_pa = pt.data_quants['erg_lepi_l2_3dflux_FPDU_pa'].sel(time=slice(*trange))    #'#/s/$cm^{2}$/str/eV'

dq_electron = pt.data_quants['erg_lepe_l2_omniflux_FEDO'].sel(time=slice(*trange))      #'#/s/$cm^{2}$/str/eV'
dq_electron_pa = pt.data_quants['erg_lepe_l2_3dflux_FEDU_pa'].sel(time=slice(*trange))  #'#/s/$cm^{2}$/str/eV'

ytitle_proton = r'LEP-i $\mathrm{H}^{+}$' + '\n' + r'[eV/q]'
ztitle_proton = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_Arase('lepi', fig, ax_1, cax_1, dq_proton, 1E0, 1E4, 4E1, 3E4, ytitle_proton, ztitle_proton)

ytitle_proton_pa = r'LEP-i $\mathrm{H}^{+}$' + '\n' + r'[deg]'
ztitle_proton_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_Arase(fig, ax_2, cax_2, dq_proton_pa, 1E0, 1E4, ytitle_proton_pa, ztitle_proton_pa)

ytitle_electron = r'LEP-e $\mathrm{e}^{-}$' + '\n' + r'[eV]'
ztitle_electron = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_Arase('lepe', fig, ax_3, cax_3, dq_electron, 1E2, 1E5, None, None, ytitle_electron, ztitle_electron)

ytitle_electron_pa = r'LEP-e $\mathrm{e}^{-}$' + '\n' + r'[deg]'
ztitle_electron_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_Arase(fig, ax_4, cax_4, dq_electron_pa, 1E2, 1E5, ytitle_electron_pa, ztitle_electron_pa)

add_panel_label(ax_1, '(b-1)')
add_panel_label(ax_2, '(b-2)')
add_panel_label(ax_3, '(b-3)')
add_panel_label(ax_4, '(b-4)')

plt.tight_layout()
plt.show()

In [ ]:
psp.themis.esa(trange=trange, probe='a', level='l2', no_update=True)

themis_a_e_omniflux = pt.data_quants['tha_peeb_en_eflux']
themis_a_e_omniflux = themis_a_e_omniflux.sel(time=slice(trange[0], trange[1]))
print(themis_a_e_omniflux)

themis_a_proton_omniflux = pt.data_quants['tha_peib_en_eflux']
themis_a_proton_omniflux = themis_a_proton_omniflux.sel(time=slice(trange[0], trange[1]))
print(themis_a_proton_omniflux)

In [ ]:
import os
import sys
import importlib
sys.path.append("..")
import module_handmade.themis_band_dnumflux as tbd
importlib.reload(tbd)

tbd.compute_band_dnumflux(
    folder=r'/mnt/j/observation_data/themis/tha/idl_output/per_bin',
    spec='peif',
    Emin=40,
    Emax=1000
)

tbd.compute_band_dnumflux(
    folder=r'/mnt/j/observation_data/themis/tha/idl_output/per_bin',
    spec='peef',
    Emin=40,
    Emax=5000
)

In [ ]:
import pandas as pd, numpy as np, xarray as xr

def csv_to_xarray_dnumflux(csv_path: str, var_name: str = "dnumflux"):
    df = pd.read_csv(csv_path)
    if "time_iso" not in df.columns:
        raise ValueError("CSVに'time_iso'列が必要です。")
    # ピッチ角中心（列名）→ spec_bins 座標
    pa_cols = [c for c in df.columns if c != "time_iso"]
    pa_vals = np.array([float(c) for c in pa_cols])  # 度

    # 時間
    time = pd.to_datetime(df["time_iso"], format="%Y-%m-%d/%H:%M:%S", errors="coerce").to_numpy()

    # データ (len(time), len(pitch_angle))
    data = df[pa_cols].to_numpy(dtype=float)

    da = xr.DataArray(
        data,
        dims=("time", "spec_bins"),
        coords={"time": time, "spec_bins": pa_vals},
        name=var_name,
        attrs={
            "units": "1/(cm^2 sr s eV)",
            "long_name": "band-averaged differential number flux",
        }
    )
    return da

da_tha_proton_pa = csv_to_xarray_dnumflux(
    r"/mnt/j/observation_data/themis/tha/idl_output/per_bin/tha_peif_dnumflux_E40to1000.csv",
    var_name="tha_peif_dnumflux_E40to1000"
    )

da_tha_electron_pa = csv_to_xarray_dnumflux(
    r"/mnt/j/observation_data/themis/tha/idl_output/per_bin/tha_peef_dnumflux_E40to5000.csv",
    var_name="tha_peef_dnumflux_E40to5000"
    )

print(da_tha_proton_pa)
print(da_tha_electron_pa)

In [ ]:
da_tha_proton = pt.data_quants['tha_peif_en_eflux']
da_tha_electron = pt.data_quants['tha_peef_en_eflux']

print(da_tha_proton)
print(da_tha_electron)

import os
import sys
import importlib
sys.path.append("..")
import module_handmade.xarray_numflux_utils as xnu
importlib.reload(xnu)

dn_tha_proton = xnu.eflux_to_numflux(da_eflux=da_tha_proton)
dn_tha_electron = xnu.eflux_to_numflux(da_eflux=da_tha_electron)

print(dn_tha_proton)
print(dn_tha_electron)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
import matplotlib.cm as cm
import pytplot as pt

def omniflux_plot_THEMIS_A(fig, ax, cax, dq, vmin=None, vmax=None, energy_min=None, energy_max=None, ytitle=None, ztitle=None):
    # 取り出し（全て (T, E) 形状）
    flux = np.asarray(dq.data, dtype=float)           # (T, E)
    E    = np.asarray(dq.spec_bins, dtype=float)      # (T, E) 変動エネルギー
    t    = np.asarray(dq.time.values)                 # (T,)

    # pcolormeshのX/Yは非有限NG → 列方向で全部有限なチャンネルだけ残す
    good_e = np.all(~np.isnan(E), axis=0)
    if not np.all(good_e):
        flux = flux[:, good_e]
        E    = E[:,    good_e]

    # ---- 値の前処理（LogNorm 用）----
    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan
    if np.all(~np.isfinite(flux)):
        raise ValueError("有効な（>0）フラックスがありません。")

    if vmin == None:
        vmin = np.nanmin(flux)
    if vmax == None:
        vmax = np.nanmax(flux)
    if not (vmin < vmax):
        vmax = vmin * 1.0001
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    #print(vmin, vmax)

    flux[np.isnan(flux)] = 1E-99

    # ---- 時間メッシュ生成（(T,E)）----
    Tmesh = np.broadcast_to(t[:, None], E.shape)

    #print(flux.shape)
    #print(E.shape)
    #print(Tmesh.shape)

    # ---- 描画 ----
    mesh = ax.pcolormesh(Tmesh, E, flux, norm=norm, cmap=cm.turbo, shading='nearest')
    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label("")  # ラベル消す
    #if ztitle:
    #    ax.text(0.85, 0.15, ztitle,
    #            transform=ax.transAxes, ha='center', va='center', color='k',
    #            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.25'))

    ax.set_yscale('log')
    ax.set_ylabel(ytitle)
    ax.grid(which='both', alpha=0.5)
    ax.minorticks_on()

    if energy_min != None:
        ax.set_ylim(ymin=energy_min)
    if energy_max != None:
        ax.set_ylim(ymax=energy_max)

    # 時刻目盛り
    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    return fig, ax, cax

def pa_plot_THEMIS_A(fig, ax, cax, dq, vmin=None, vmax=None, ytitle=None, ztitle=None):
    # 取り出し（全て (T, E) 形状）
    flux = np.asarray(dq.data, dtype=float)           # (T, E)
    PA    = np.asarray(dq.spec_bins, dtype=float)     # (E)
    PA    = np.broadcast_to(PA[None, :], flux.shape)  # (T, E)
    t    = np.asarray(dq.time.values)                 # (T,)
    print(flux)

    # pcolormeshのX/Yは非有限NG → 列方向で全部有限なチャンネルだけ残す
    good_PA = np.all(~np.isnan(PA), axis=0)
    if not np.all(good_PA):
        flux = flux[:, good_PA]
        PA    = PA[:,    good_PA]

    # ---- 値の前処理（LogNorm 用）----
    flux[~np.isfinite(flux)] = np.nan
    flux[flux <= 0] = np.nan
    if np.all(~np.isfinite(flux)):
        raise ValueError("有効な（>0）フラックスがありません。")

    if vmin == None:
        vmin = np.nanmin(flux)
    if vmax == None:
        vmax = np.nanmax(flux)
    if not (vmin < vmax):
        vmax = vmin * 1.0001
    norm = mcolors.LogNorm(vmin=vmin, vmax=vmax)
    #print(vmin, vmax)

    flux[np.isnan(flux)] = 1E-99

    # ---- 時間メッシュ生成（(T,E)）----
    print(t.shape)
    if t.shape == flux.shape:
        Tmesh = t
    else:
        Tmesh = np.broadcast_to(t[:, None], flux.shape)

    #print(flux.shape)
    #print(PA.shape)
    #print(Tmesh.shape)

    # ---- 描画 ----
    mesh = ax.pcolormesh(Tmesh, PA, flux, norm=norm, cmap=cm.turbo, shading='nearest')
    cb = fig.colorbar(mesh, cax=cax)
    cb.set_label("")  # ラベル消す
    #if ztitle:
    #    ax.text(0.85, 0.5, ztitle,
    #            transform=ax.transAxes, ha='center', va='center', color='k',
    #            bbox=dict(facecolor='white', alpha=0.5, boxstyle='round,pad=0.25'))

    ax.set_ylabel(ytitle)
    ax.grid(which='both', alpha=0.5)
    ax.minorticks_on()

    ax.set_ylim(ymax=180, ymin=0)
    ax.set_yticks(np.arange(0, 181, 45))

    # 時刻目盛り
    loc = mdates.AutoDateLocator()
    fmt = mdates.ConciseDateFormatter(loc)
    ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(fmt)

    return fig, ax, cax


fig = plt.figure(figsize=(10, 12))
gs = fig.add_gridspec(4, 2, width_ratios=[1, 0.025], wspace=0.05)
ax_1 = fig.add_subplot(gs[0, 0])
cax_1 = fig.add_subplot(gs[0, 1])
ax_2 = fig.add_subplot(gs[1, 0], sharex=ax_1)
cax_2 = fig.add_subplot(gs[1, 1])
ax_3 = fig.add_subplot(gs[2, 0], sharex=ax_1)
cax_3 = fig.add_subplot(gs[2, 1])
ax_4 = fig.add_subplot(gs[3, 0], sharex=ax_1)
cax_4 = fig.add_subplot(gs[3, 1])

ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)

dn_tha_proton = dn_tha_proton.sel(time=slice(*trange))
dn_tha_electron = dn_tha_electron.sel(time=slice(*trange))
da_tha_proton_pa = da_tha_proton_pa.sel(time=slice(*trange))
da_tha_electron_pa = da_tha_electron_pa.sel(time=slice(*trange))

ytitle_proton = r'ESA ion' + '\n' + r'[eV/q]'
ztitle_proton = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_THEMIS_A(fig, ax_1, cax_1, dn_tha_proton, 1E0, 1E4, None, None, ytitle_proton, ztitle_proton)

ytitle_proton_pa = r'ESA ion' + '\n' + r'[deg]'
ztitle_proton_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_THEMIS_A(fig, ax_2, cax_2, da_tha_proton_pa, 1E0, 1E4, ytitle_proton_pa, ztitle_proton_pa)

ytitle_electron = r'ESA $\mathrm{e}^{-}$' + '\n' + r'[eV]'
ztitle_electron = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_THEMIS_A(fig, ax_3, cax_3, dn_tha_electron, 1E2, 1E5, None, None, ytitle_electron, ztitle_electron)

ytitle_electron_pa = r'ESA $\mathrm{e}^{-}$' + '\n' + r'[deg]'
ztitle_electron_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_THEMIS_A(fig, ax_4, cax_4, da_tha_electron_pa, 1E2, 1E5, ytitle_electron_pa, ztitle_electron_pa)

add_panel_label(ax_1, '(c-1)')
add_panel_label(ax_2, '(c-2)')
add_panel_label(ax_3, '(c-3)')
add_panel_label(ax_4, '(c-4)')

plt.tight_layout()
plt.show()


# $|\bf{E}_{\perp}|$、$|\bf{B}_{\perp}|$、$S_{\parallel}$のplot

In [ ]:
import pyspedas as psp
import pytplot as pt
import numpy as np
import xarray as xr

pt.del_data('*')

time_range = ['20220901/21:00:00', '20220902/00:00:00']

psp.erg.pwe_efd(trange=time_range, level='l2', datatype='64', coord='dsi', no_update=True)
psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', coord='dsi', no_update=True)
psp.erg.orb(trange=time_range, level='l2', no_update=True)

E64_data_Ex = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ex_waveform']
E64_data_Ey = pt.data_quants['erg_pwe_efd_l2_E64Hz_dsi_Ey_waveform']
B64_data    = pt.data_quants['erg_mgf_l2_mag_64hz_dsi']

time_range_T = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
E64_data_Ex = E64_data_Ex.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
E64_data_Ey = E64_data_Ey.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B64_data    = B64_data.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

# --- 0. 準備：変数名は質問に合わせている -----------------------------
Ex = E64_data_Ex.sortby('time')           # 時系列を昇順に
Ey = E64_data_Ey.sortby('time')
B  = B64_data.sortby('time')              # 64 Hz 磁場ベクトル

Bx = B.isel(v_dim=0)         # dims = ('time',)
By = B.isel(v_dim=1)
Bz = B.isel(v_dim=2)

# --- 1. E を B のタイムスタンプへ線形補間 ----------------------------
Ex_i = Ex.interp(time=B.time, method='linear')
Ey_i = Ey.interp(time=B.time, method='linear')
#   • 範囲外はデフォルトで NaN。必要なら
#     .interp(..., kwargs={'fill_value': 'extrapolate'}) など

# --- 2. Ez を計算 ----------------------------------------------------
EPS = 1e-12          # ゼロ割り回避用の閾値 (単位: nT)
Ez = xr.where(np.abs(Bz) > EPS, -(Ex_i*Bx + Ey_i*By)/Bz, np.nan)
Ez.name = 'E64Hz_dsi_Ez_waveform'

# --- 3. 必要なら Dataset にまとめる --------------------------------
ds = xr.Dataset({
    'Ex_dsi': Ex_i,
    'Ey_dsi': Ey_i,
    'Ez_dsi': Ez,
    'Bx_dsi': ('time', Bx.data),
    'By_dsi': ('time', By.data),
    'Bz_dsi': ('time', Bz.data),
}, coords={'time': B.time})

ds = ds.dropna(dim='time', how='any', subset=['Ex_dsi', 'Ey_dsi', 'Bx_dsi', 'By_dsi', 'Bz_dsi'])


def dsi_to_fac(ds: xr.Dataset, window_sec: float = 100.0) -> xr.Dataset:
    """DSI→FAC 変換 (64 Hz データを想定)
    Parameters
    ----------
    ds : xr.Dataset
        必須変数: Ex_dsi, Ey_dsi, Ez_dsi, Bx_dsi, By_dsi, Bz_dsi
    window_sec : float, optional
        FAC 基底を決める移動平均時間 [s]
    Returns
    -------
    ds_out : xr.Dataset
        DSI + FAC の両方を含む Dataset
    """
    # --- 0. 時系列を昇順にしておく -----------------------------------
    ds = ds.sortby('time')

    # --- 1. 平均磁場 <B> の計算 --------------------------------------
    dt = (ds.time[1] - ds.time[0]).astype('timedelta64[ns]').astype(float) * 1e-9
    win_pts = int(window_sec / dt)
    B_dsi = ds[['Bx_dsi', 'By_dsi', 'Bz_dsi']].to_array('comp')      # (comp,time)
    print(B_dsi)

    B_avg = B_dsi.rolling(time=win_pts, center=True).mean('time')          # (comp,time)
    B_avg = B_avg.transpose('time', 'comp').values                   # (N,3)

    # --- 2. FAC 基底ベクトル -----------------------------------------
    z0 = np.array([0.0, 0.0, 1.0])
    e2 = np.cross(z0, B_avg)                                              # ⟂B & ⟂z
    # z0 // B のとき e2≈0 → 代わりに x軸とクロスする等の処理を追加しても良い
    e1 = np.cross(e2, B_avg)
    e3 = B_avg

    def _normalize(v):
        return v / np.linalg.norm(v, axis=1, keepdims=True)

    e1_hat = _normalize(e1)
    e2_hat = _normalize(e2)
    e3_hat = _normalize(e3)

    # --- 3. 回転行列 R (time,3,3) とベクトル変換 ----------------------
    R = np.stack([e1_hat, e2_hat, e3_hat], axis=2)                   # columns=ê_i

    # DSI→FAC: v_fac = R · v_dsi
    E_dsi = ds[['Ex_dsi', 'Ey_dsi', 'Ez_dsi']].to_array('comp') \
              .transpose('time', 'comp').values                      # (N,3)
    B_dsi_np = B_dsi.transpose('time', 'comp').values               # (N,3)

    E_fac = np.einsum('tji,tj->ti', R, E_dsi)                        # (N,3)
    B_fac = np.einsum('tji,tj->ti', R, B_dsi_np)                     # (N,3)

    # --- 4. Dataset へ格納 ------------------------------------------
    ds_out = ds.copy()
    ds_out['Ex_fac'], ds_out['Ey_fac'], ds_out['Ez_fac'] = [
        (('time'), E_fac[:, k]) for k in range(3)]
    ds_out['Bx_fac'], ds_out['By_fac'], ds_out['Bz_fac'] = [
        (('time'), B_fac[:, k]) for k in range(3)]

    # attrs など必要ならここで設定
    return ds_out, R, e3_hat


# ---------- 使い方 -------------------------------------------------------
ds_fac, Rotation_tensor, e3_hat = dsi_to_fac(ds)

ds_fac = ds_fac.sel(time=slice(*trange))

In [ ]:
Eperp_2 = ((ds_fac['Ex_fac'].data)**2E0 + (ds_fac['Ey_fac'].data)**2E0)
Bperp_2 = ((ds_fac['Bx_fac'].data)**2E0 + (ds_fac['By_fac'].data)**2E0)

mu_0 = 4.*np.pi * 1E-7

S_para = (ds_fac['Ex_fac'].data * ds_fac['By_fac'].data - ds_fac['Ey_fac'].data * ds_fac['Bx_fac'].data) / mu_0 * 1E-12 #W/m^2

In [ ]:
import matplotlib.ticker as mticker
from datetime import datetime

mpl.rcParams['font.size'] = 25

fig = plt.figure(figsize=(10, 22))

gs = fig.add_gridspec(7, 2, width_ratios=[1, 0.025], wspace=0.05, hspace=0.15)
ax_1 = fig.add_subplot(gs[0, 0])
cax_1 = fig.add_subplot(gs[0, 1])
ax_2 = fig.add_subplot(gs[1, 0], sharex=ax_1)
cax_2 = fig.add_subplot(gs[1, 1])
ax_3 = fig.add_subplot(gs[2, 0], sharex=ax_1)
cax_3 = fig.add_subplot(gs[2, 1])
ax_4 = fig.add_subplot(gs[3, 0], sharex=ax_1)
cax_4 = fig.add_subplot(gs[3, 1])
ax_5 = fig.add_subplot(gs[4, 0], sharex=ax_1)
ax_6 = fig.add_subplot(gs[5, 0], sharex=ax_1)
ax_7 = fig.add_subplot(gs[6, 0], sharex=ax_1)

ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)
ax_4.tick_params(axis='x', which='both', labelbottom=False)
ax_5.tick_params(axis='x', which='both', labelbottom=False)
ax_6.tick_params(axis='x', which='both', labelbottom=False)

ytitle_proton = r'LEP-i $\mathrm{H}^{+}$' + '\n' + r'[eV/q]'
ztitle_proton = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_Arase('lepi', fig, ax_1, cax_1, dq_proton, 1E1, 1E3, 4E1, 1E4, ytitle_proton, ztitle_proton)

ytitle_proton_pa = r'LEP-i $\mathrm{H}^{+}$' + '\n' + r'[deg]'
ztitle_proton_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_Arase(fig, ax_2, cax_2, dq_proton_pa, 1E1, 1E3, ytitle_proton_pa, ztitle_proton_pa)

ytitle_electron = r'LEP-e $\mathrm{e}^{-}$' + '\n' + r'[eV]'
ztitle_electron = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_Arase('lepe', fig, ax_3, cax_3, dq_electron, 1E3, 1E5, 4E1, 1E4, ytitle_electron, ztitle_electron)

ytitle_electron_pa = r'LEP-e $\mathrm{e}^{-}$' + '\n' + r'[deg]'
ztitle_electron_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_Arase(fig, ax_4, cax_4, dq_electron_pa, 1E3, 1E5, ytitle_electron_pa, ztitle_electron_pa)

ax_5.plot(ds_fac.time, Eperp_2*1E-4, c='k', linewidth=0.5)
ax_5.set_ylabel(r'$|\mathbf{E}_{\perp}|^{2}$' + '\n' + r'[$10^{4} (\mathrm{mV/m})^{2}$]')
ax_5.set_ylim(ymin=0, ymax=4)
ax_5.minorticks_on()
ax_5.grid(which='both', alpha=0.5)

ax_6.plot(ds_fac.time, Bperp_2, c='k', linewidth=0.5)
ax_6.set_ylabel(r'$|\mathbf{B}_{\perp}|^{2}$' + '\n' + r'[$(\mathrm{nT})^{2}$]')
ax_6.set_ylim(ymin=0)
ax_6.minorticks_on()
ax_6.grid(which='both', alpha=0.5)

ax_7.plot(ds_fac.time, S_para*1E3, c='k', linewidth=0.5)
ax_7.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#ax_7.set_yscale('symlog')
ax_7.minorticks_on()
ax_7.grid(which='both', alpha=0.5)


time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_7.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax_7.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax_7.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

def add_panel_label(ax, label, x=-0.15, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

def to_py_datetime(t_np64):
    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)

# 軌道データ
pos_da = pt.data_quants['erg_orb_l2_pos_rmlatmlt']  # (Nt, 3)
t_pos_py = to_py_datetime(pos_da.time.values)
t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数

R   = np.asarray(pos_da.values[:, 0], dtype=float)  # Re
mlat= np.asarray(pos_da.values[:, 1], dtype=float)  # deg
mlt = np.asarray(pos_da.values[:, 2], dtype=float)  # hour [0,24)

# --- MLT の 24h 周期をほどいてから補間し、最後に 24 で折り返す ---
mlt_unwrap = np.unwrap(mlt * 2*np.pi/24.0) * 24.0/(2*np.pi)

# 補間関数（tick の x は「日数」なのでそのまま使う）
def interp_at(x_num):
    Ri    = np.interp(x_num, t_pos_num, R, left=np.nan, right=np.nan)
    mlati = np.interp(x_num, t_pos_num, mlat, left=np.nan, right=np.nan)
    mltiu = np.interp(x_num, t_pos_num, mlt_unwrap, left=np.nan, right=np.nan)
    mlti  = np.mod(mltiu, 24.0)
    return Ri, mlati, mlti

# 目盛フォーマッタ
def rmlt_formatter(x, pos=None):
    Ri, mlati, mlti = interp_at(x)
    if np.any(~np.isfinite([Ri, mlati, mlti])):
        return ""  # 範囲外は空
    return (f"{Ri:0.2f}\n"
            f"{mlati:0.2f}\n"
            f"{mlti:0.2f}")

# セカンダリ x 軸（底 side）を作ってラベルを差し替え
secax = ax_7.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
secax.xaxis.set_major_formatter(mticker.FuncFormatter(rmlt_formatter))

# メインの時間ラベルと重ならないよう余白を広げる
ax_7.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル

# 好みで：目盛間隔をメイン x と合わせる
secax.set_ticks(ax_7.get_xticks())

fig.text(0.01, 0.100, "hhmm", ha='center', va='center')
fig.text(0.01, 0.083, r"R [$R_{\mathrm{E}}$]", ha='center', va='center')
fig.text(0.01, 0.063, r"MLAT", ha='center', va='center')
fig.text(0.01, 0.045, r"MLT", ha='center', va='center')

add_panel_label(ax_1, '(b-1)')
add_panel_label(ax_2, '(b-2)')
add_panel_label(ax_3, '(b-3)')
add_panel_label(ax_4, '(b-4)')
add_panel_label(ax_5, '(b-5)')
add_panel_label(ax_6, '(b-6)')
add_panel_label(ax_7, '(b-7)')

fig.suptitle('Arase', y=0.9)

fig.tight_layout()
plt.show()

#fig.savefig(r"/mnt/j/KAW_observation/Figure_1_b.pdf", bbox_inches='tight')

In [ ]:
pos_da_analysis = pos_da.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

print(np.nanmin(pos_da_analysis[:, 0]), np.nanmax(pos_da_analysis[:, 0]))
print(np.nanmin(pos_da_analysis[:, 1]), np.nanmax(pos_da_analysis[:, 1]))
print(np.nanmin(pos_da_analysis[:, 2]), np.nanmax(pos_da_analysis[:, 2]))


In [ ]:
import pyspedas as psp
import pytplot as pt


path_base_save_plot = f'/mnt/j/KAW_observation/E_B_ratio_THEMIS_A/2022-09-01/2230-2330_low_freq'

psp.themis.fgm(trange=time_range, probe='a', level='l2', no_update=True)                    # fgh: 128 Hz, fgl: 16 Hz, fgs: 2.74 sec
#psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efw')    # efw: 16448 Hz
#psp.themis.efi(trange=time_range, probe='a', level='l2', datatype='efp')    # efp: 512 Hz
psp.themis.efi(trange=time_range, probe='a', level='l2', no_update=True)                    # eff: 8 Hz


E8_data_gsm     = pt.data_quants['tha_eff_dot0_gsm']
B16_data_gsm    = pt.data_quants['tha_fgl_gsm']

time_range_T    = [time_range[0].replace('/', 'T'), time_range[1].replace('/', 'T')]
E8_data_gsm   = E8_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))
B16_data_gsm   = B16_data_gsm.sortby('time').sel(time=slice(time_range_T[0], time_range_T[1]))

E8_data_gsm_x = E8_data_gsm.isel(v_dim=0)
E8_data_gsm_y = E8_data_gsm.isel(v_dim=1)
E8_data_gsm_z = E8_data_gsm.isel(v_dim=2)

B16_data_gsm_x = B16_data_gsm.isel(v_dim=0)
B16_data_gsm_y = B16_data_gsm.isel(v_dim=1)
B16_data_gsm_z = B16_data_gsm.isel(v_dim=2)

pt.store_data('E8_gsm_x', data={'x': E8_data_gsm_x.time, 'y': E8_data_gsm_x})
pt.store_data('E8_gsm_y', data={'x': E8_data_gsm_x.time, 'y': E8_data_gsm_x})
pt.store_data('E8_gsm_z', data={'x': E8_data_gsm_x.time, 'y': E8_data_gsm_x})
pt.store_data('B16_gsm_x', data={'x': B16_data_gsm_x.time, 'y': B16_data_gsm_x})
pt.store_data('B16_gsm_y', data={'x': B16_data_gsm_y.time, 'y': B16_data_gsm_y})
pt.store_data('B16_gsm_z', data={'x': B16_data_gsm_z.time, 'y': B16_data_gsm_z})

gap_thr = np.timedelta64(30, 's')

# ---------- 1. 電場チャンク ----------------------------------
t_ref_E = pt.data_quants['E8_gsm_x'].time.values
is_new_E = np.concatenate(([True], np.diff(t_ref_E) > gap_thr))
chunk_E  = np.cumsum(is_new_E) - 1

# ---------- 2. 磁場チャンク ----------------------------------
t_ref_B = pt.data_quants['B16_gsm_x'].time.values
is_new_B = np.concatenate(([True], np.diff(t_ref_B) > gap_thr))
chunk_B  = np.cumsum(is_new_B) - 1

# ---------- 3. 分割ループ ------------------------------------
vars_E = ['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z']
vars_B = ['B16_gsm_x', 'B16_gsm_y', 'B16_gsm_z']

def split_and_store(var_list, t_ref, chunk_id):
    for v in var_list:
        da   = pt.data_quants[v]
        for k in np.unique(chunk_id):
            sel = chunk_id == k
            if sel.sum() == 0:
                continue
            new = f'{v}_seg{k}'
            pt.store_data(new,
                          data={'x': t_ref[sel], 'y': da.values[sel]})
            unit = '[mV/m]' if v.startswith('e_') else '[nT]'
            pt.options(new,'ytitle',v); pt.options(new,'ysubtitle',unit)

split_and_store(vars_E, t_ref_E, chunk_E)
split_and_store(vars_B, t_ref_B, chunk_B)

# ---------- 4. まとめ変数 ------------------------------------
def make_all(base):
    pt.store_data(f'{base}_all', data=pt.tnames(f'{base}_seg*'))

for base in vars_E + vars_B:
    make_all(base)

import sys
sys.path.append("..")
import module_handmade.psd_plotter_themis as pp
import importlib
importlib.reload(pp)

# ────────────────────────────────────────────────
# 0. セグメント名リストを取得
# ────────────────────────────────────────────────
seg_ids = sorted({v.split('_seg')[-1]                      # {'0','1',...}
                  for v in pt.tnames('E8_gsm_x_seg*')})

# まとめ用リスト（後で overplot するために使う）
ex_all, ey_all, ez_all = [], [], []
bx_all, by_all, bz_all = [], [], []

seg_ids_B = sorted({v.split('_seg')[-1]                      # {'0','1',...}
                  for v in pt.tnames('B16_gsm_x_seg*')})

B_x_concat = pp.concat_tplot_segments('B16_gsm_x', seg_ids_B)
B_y_concat = pp.concat_tplot_segments('B16_gsm_y', seg_ids_B)
B_z_concat = pp.concat_tplot_segments('B16_gsm_z', seg_ids_B)

ds_B = xr.Dataset({'B16_gsm_x': B_x_concat, 'B16_gsm_y': B_y_concat, 'B16_gsm_z': B_z_concat}, coords={'time': B_x_concat.time})

print(ds_B)

for sid in seg_ids:
    # TVar 取り出し → pandas DataFrame へ （xarray に変換するため）
    def to_df(var):           # var = 'e_per1_seg0' など
        dq = pt.data_quants[var]
        return pd.DataFrame({'time': dq.time.values,
                             var.split('_seg')[0]: dq.values})

    df_E = (to_df(f'E8_gsm_x_seg{sid}')
            .merge(to_df(f'E8_gsm_y_seg{sid}'), on='time')
            .merge(to_df(f'E8_gsm_z_seg{sid}'),  on='time'))

    # ---- xarray Dataset へ ----
    ds_E = xr.Dataset({k:(('time',), df_E[k].to_numpy(dtype=float))
                       for k in ['E8_gsm_x','E8_gsm_y','E8_gsm_z']},
                      coords={'time': df_E['time'].to_numpy('datetime64[ns]')})

    rename_dict_B = {
        'B16_gsm_x': 'B8_gsm_x',
        'B16_gsm_y': 'B8_gsm_y',
        'B16_gsm_z': 'B8_gsm_z'
    }

    # renameメソッドで変数名を変更 (16 Hz -> 8 Hz)
    ds_B_16 = ds_B.rename(rename_dict_B)

    # ---- B を E の時刻へ線形補間 ----
    ds_B_8  = ds_B_16.interp(time=ds_E.time, method='linear')
    ds_merged = xr.merge([ds_E, ds_B_8])
    ds_merged = ds_merged.dropna(dim='time', how='any', subset=['E8_gsm_x', 'E8_gsm_y', 'E8_gsm_z', 'B8_gsm_x', 'B8_gsm_y', 'B8_gsm_z'])

    # ---- TVar 登録（seg ID を引き継ぐ） ----
    for var in ds_merged.data_vars:
        new_name = f'{var}_i_seg{sid}'
        pt.store_data(new_name,
                      data={'x': ds_merged.time.values,
                            'y': ds_merged[var].values})
        unit = '[mV/m]' if var.startswith('E8_') else '[nT]'
        pt.options(new_name, 'ytitle', var)
        pt.options(new_name, 'ysubtitle', unit)

        # まとめリストへ追加
        if   var == 'E8_gsm_x': ex_all.append(new_name)
        elif var == 'E8_gsm_y': ey_all.append(new_name)
        elif var == 'E8_gsm_z': ez_all.append(new_name)
        elif var == 'B8_gsm_x': bx_all.append(new_name)
        elif var == 'B8_gsm_y': by_all.append(new_name)
        elif var == 'B8_gsm_z': bz_all.append(new_name)

def rotate_segment_to_fac(sid, vec_base_name, matrix_da):
    """
    指定されたセグメントのベクトルデータをFACへ変換し、成分に分割する。
    戻り値: 3つの成分変数名のリスト [comp_x, comp_y, comp_z]
    """
    print(f"  - Rotating vector: {vec_base_name}_i_seg{sid}")
    
    # 1. ベクトルデータを取得
    vec_tvar = f'{vec_base_name}_i_seg{sid}'
    t_vec, d_vec = pt.get_data(vec_tvar)

    # 1a. pandasを使って時間軸の重複をチェックし、削除する
    #     重複があった場合、最初の値を採用する (keep='first')
    unique_indices = pd.Index(t_vec).is_unique
    if not unique_indices:
        print(f"    - 警告: {vec_tvar} の時間軸に重複が見つかりました。重複を削除します。")
        _, unique_idx = np.unique(t_vec, return_index=True)
        t_vec = t_vec[unique_idx]
        d_vec = d_vec[unique_idx]

    # 2. 回転行列をベクトルの時間軸に手動で補間
    matrix_interp = matrix_da.interp(time_mat=t_vec, method="linear").values

    # 3. NumPyのeinsumで手動でベクトル回転
    fac_np = np.einsum('tij,tj->ti', matrix_interp, d_vec)
    
    prefix = 'E' if 'E' in vec_base_name else 'B'
    fac_tvar = f'{prefix}_fac_vec_seg{sid}'
    pt.store_data(fac_tvar, data={'x': t_vec, 'y': fac_np})
    
    default_component_names = psp.split_vec(fac_tvar) 
    desired_component_names = [f'{prefix}{c}_fac_seg{sid}' for c in ['x', 'y', 'z']]
    for i in range(3):
        pt.tplot_rename(default_component_names[i], desired_component_names[i])
    
    return desired_component_names

# ------------------------------------------------------------
# 0. パラメータ設定
# ------------------------------------------------------------
b_field_lf_tvar = 'tha_fgs_gsm'
e_field_hf_base = 'E8_gsm'
b_field_hf_base = 'B8_gsm'
seg_ids = sorted({v.split('_seg')[-1] for v in pt.tnames(f'{e_field_hf_base}_x_i_seg*')})

# 100 sec rolling
B_rolling = pt.data_quants[b_field_lf_tvar]
dt = (B_rolling.time.values[1] - B_rolling.time.values[0]).astype('timedelta64[ns]').astype(float) * 1e-9
win_secs = 100.0
win_pts = int(win_secs / dt)
B_rolling = B_rolling.rolling(time=win_pts, center=True).mean()
b_field_lf_tvar_rolling = b_field_lf_tvar + '_rolling'
print(b_field_lf_tvar_rolling)
pt.store_data(b_field_lf_tvar_rolling, data={'x': B_rolling.time, 'y': B_rolling.values}, attr_dict=B_rolling.attrs)

# ------------------------------------------------------------
# 1. 成分データをベクトルに結合し、メタデータを設定
# ------------------------------------------------------------
print("--- ステップ1: 成分データをベクトルに結合し、座標系メタデータを設定 ---")
for sid in seg_ids:
    e_components = [f'{e_field_hf_base}_x_i_seg{sid}', f'{e_field_hf_base}_y_i_seg{sid}', f'{e_field_hf_base}_z_i_seg{sid}']
    e_vec_tvar = f'{e_field_hf_base}_i_seg{sid}'
    pt.join_vec(e_components, newname=e_vec_tvar)
    pt.data_quants[e_vec_tvar].attrs['coordinate_system'] = 'gsm'
    
    b_components = [f'{b_field_hf_base}_x_i_seg{sid}', f'{b_field_hf_base}_y_i_seg{sid}', f'{b_field_hf_base}_z_i_seg{sid}']
    b_vec_tvar = f'{b_field_hf_base}_i_seg{sid}'
    pt.join_vec(b_components, newname=b_vec_tvar)
    pt.data_quants[b_vec_tvar].attrs['coordinate_system'] = 'gsm'
print("ベクトル変数の作成とメタデータの設定が完了しました。")

# ------------------------------------------------------------
# 2. FAC回転行列の作成と準備
# ------------------------------------------------------------
print("\n--- ステップ2: FAC回転行列を作成し、準備 ---")
fac_matrix_tvar = psp.fac_matrix_make(b_field_lf_tvar_rolling)

t_mat, d_mat = pt.get_data(fac_matrix_tvar)

if t_mat is not None:
    unique_indices_mat = pd.Index(t_mat).is_unique
    if not unique_indices_mat:
        print(f"警告: 回転行列 ({fac_matrix_tvar}) の時間軸に重複が見つかりました。重複を削除します。")
        _, unique_idx = np.unique(t_mat, return_index=True)
        t_mat = t_mat[unique_idx]
        # d_matも同じインデックスでスライスして、時間とデータの整合性を保つ
        d_mat = d_mat[unique_idx]

if d_mat is not None and d_mat.ndim == 2:
    d_mat = d_mat[np.newaxis, :, :]
matrix_da = xr.DataArray(d_mat, dims=('time_mat', 'row', 'col'), coords={'time_mat': t_mat})
print(f"回転行列をxarray.DataArrayとして準備完了。Shape: {matrix_da.shape}")

# ------------------------------------------------------------
# 3. segment ごとに FAC 変換 (補助関数を利用)
# ------------------------------------------------------------
print("\n--- ステップ3: 各セグメントをFACへ変換 ---")
fac_vars = {'Ex': [], 'Ey': [], 'Ez': [], 'Bx': [], 'By': [], 'Bz': []}

for sid in seg_ids:
    print(f"\n--- Processing segment {sid} ---")
    
    # 補助関数を呼び出して電場と磁場をそれぞれ変換
    e_fac_components = rotate_segment_to_fac(sid, e_field_hf_base, matrix_da)
    b_fac_components = rotate_segment_to_fac(sid, b_field_hf_base, matrix_da)
    
    # 結果をリストに格納
    fac_vars['Ex'].append(e_fac_components[0])
    fac_vars['Ey'].append(e_fac_components[1])
    fac_vars['Ez'].append(e_fac_components[2])
    fac_vars['Bx'].append(b_fac_components[0])
    fac_vars['By'].append(b_fac_components[1])
    fac_vars['Bz'].append(b_fac_components[2])

pt.store_data('Ex_fac_all', data=fac_vars['Ex'])
pt.store_data('Ey_fac_all', data=fac_vars['Ey'])
pt.store_data('Ez_fac_all', data=fac_vars['Ez'])
pt.store_data('Bx_fac_all', data=fac_vars['Bx'])
pt.store_data('By_fac_all', data=fac_vars['By'])
pt.store_data('Bz_fac_all', data=fac_vars['Bz'])

In [ ]:
Ex_fac_vec = xr.concat([pt.data_quants['Ex_fac_seg0'], pt.data_quants['Ex_fac_seg1'], pt.data_quants['Ex_fac_seg2']], dim='time')
Ey_fac_vec = xr.concat([pt.data_quants['Ey_fac_seg0'], pt.data_quants['Ey_fac_seg1'], pt.data_quants['Ey_fac_seg2']], dim='time')
Ez_fac_vec = xr.concat([pt.data_quants['Ez_fac_seg0'], pt.data_quants['Ez_fac_seg1'], pt.data_quants['Ez_fac_seg2']], dim='time')
Bx_fac_vec = xr.concat([pt.data_quants['Bx_fac_seg0'], pt.data_quants['Bx_fac_seg1'], pt.data_quants['Bx_fac_seg2']], dim='time')
By_fac_vec = xr.concat([pt.data_quants['By_fac_seg0'], pt.data_quants['By_fac_seg1'], pt.data_quants['By_fac_seg2']], dim='time')
Bz_fac_vec = xr.concat([pt.data_quants['Bz_fac_seg0'], pt.data_quants['Bz_fac_seg1'], pt.data_quants['Bz_fac_seg2']], dim='time')

In [ ]:
Eperp_2_THA = ((Ex_fac_vec.data)**2E0 + (Ey_fac_vec.data)**2E0)
Bperp_2_THA = ((Bx_fac_vec.data)**2E0 + (By_fac_vec.data)**2E0)

mu_0 = 4.*np.pi * 1E-7

S_para_THA = (Ex_fac_vec.data * By_fac_vec.data - Ey_fac_vec.data * Bx_fac_vec.data) / mu_0 * 1E-12 #W/m^2

print(Eperp_2_THA)
print(Bperp_2_THA)
print(S_para_THA)

In [ ]:
psp.themis.state(trange=trange, probe='a', no_update=True)

In [ ]:
psp.cotrans(name_in='tha_pos_gsm', name_out='tha_pos_sm', coord_in='gsm', coord_out='sm')
THA_SM_pos = pt.data_quants['tha_pos_sm'].sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
print(THA_SM_pos)

THA_rmlatmlt_R = np.sqrt(THA_SM_pos.data[:, 0]**2E0 + THA_SM_pos.data[:, 1]**2E0 + THA_SM_pos.data[:, 2]**2E0) / 6378.1
THA_rmlatmlt_MLAT = np.rad2deg(np.arctan2(THA_SM_pos.data[:, 2], np.sqrt(THA_SM_pos.data[:, 0]**2E0 + THA_SM_pos.data[:, 1]**2E0)))
THA_rmlatmlt_MLT = np.rad2deg(np.arctan2(THA_SM_pos.data[:, 1], THA_SM_pos.data[:, 0])) / 15. + 12.

print(np.nanmax(THA_rmlatmlt_R), np.nanmin(THA_rmlatmlt_R))
print(np.nanmax(THA_rmlatmlt_MLAT), np.nanmin(THA_rmlatmlt_MLAT))
print(np.nanmax(THA_rmlatmlt_MLT), np.nanmin(THA_rmlatmlt_MLT))

In [ ]:
import matplotlib.ticker as mticker

fig = plt.figure(figsize=(10, 22))
gs = fig.add_gridspec(7, 2, width_ratios=[1, 0.025], wspace=0.05, hspace=0.15)
ax_1 = fig.add_subplot(gs[0, 0])
cax_1 = fig.add_subplot(gs[0, 1])
ax_2 = fig.add_subplot(gs[1, 0], sharex=ax_1)
cax_2 = fig.add_subplot(gs[1, 1])
ax_3 = fig.add_subplot(gs[2, 0], sharex=ax_1)
cax_3 = fig.add_subplot(gs[2, 1])
ax_4 = fig.add_subplot(gs[3, 0], sharex=ax_1)
cax_4 = fig.add_subplot(gs[3, 1])
ax_5 = fig.add_subplot(gs[4, 0], sharex=ax_1)
ax_6 = fig.add_subplot(gs[5, 0], sharex=ax_1)
ax_7 = fig.add_subplot(gs[6, 0], sharex=ax_1)

ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)
ax_4.tick_params(axis='x', which='both', labelbottom=False)
ax_5.tick_params(axis='x', which='both', labelbottom=False)
ax_6.tick_params(axis='x', which='both', labelbottom=False)

ytitle_proton = r'ESA ion' + '\n' + r'[eV]'
ztitle_proton = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_THEMIS_A(fig, ax_1, cax_1, dn_tha_proton, 1E1, 1E3, 4E1, 1E4, ytitle_proton, ztitle_proton)

ytitle_proton_pa = r'ESA ion' + '\n' + r'[deg]'
ztitle_proton_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_THEMIS_A(fig, ax_2, cax_2, da_tha_proton_pa, 1E1, 1E3, ytitle_proton_pa, ztitle_proton_pa)

ytitle_electron = r'ESA $\mathrm{e}^{-}$' + '\n' + r'[eV]'
ztitle_electron = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
omniflux_plot_THEMIS_A(fig, ax_3, cax_3, dn_tha_electron, 1E3, 1E5, 4E1, 1E4, ytitle_electron, ztitle_electron)

ytitle_electron_pa = r'ESA $\mathrm{e}^{-}$' + '\n' + r'[deg]'
ztitle_electron_pa = r'[$\mathrm{cm}^{-2} \mathrm{sr}^{-1} \mathrm{s}^{-1} \mathrm{eV}^{-1}$]'
pa_plot_THEMIS_A(fig, ax_4, cax_4, da_tha_electron_pa, 1E3, 1E5, ytitle_electron_pa, ztitle_electron_pa)


ax_5.plot(Ex_fac_vec.time, Eperp_2_THA, c='k', linewidth=0.5)
ax_5.set_ylabel(r'$|\mathbf{E}_{\perp}|^{2}$' + '\n' + r'[$(\mathrm{mV/m})^{2}$]')
ax_5.set_ylim(ymin=0)#, ymax=4)
ax_5.minorticks_on()
ax_5.grid(which='both', alpha=0.5)

ax_6.plot(Ex_fac_vec.time, Bperp_2_THA, c='k', linewidth=0.5)
ax_6.set_ylabel(r'$|\mathbf{B}_{\perp}|^{2}$' + '\n' + r'[$(\mathrm{nT})^{2}$]')
ax_6.set_ylim(ymin=0)
ax_6.minorticks_on()
ax_6.grid(which='both', alpha=0.5)

ax_7.plot(Ex_fac_vec.time, S_para_THA*1E3, c='k', linewidth=0.5)
ax_7.set_ylabel(r'$S_{\parallel}$' + '\n' + r'[$\mathrm{mW/m^{2}}$]')
#ax_7.set_yscale('symlog')
ax_7.minorticks_on()
ax_7.grid(which='both', alpha=0.5)


time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]
to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_7.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax_7.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
ax_7.xaxis.set_major_locator(mdates.MinuteLocator(interval=10))

def add_panel_label(ax, label, x=-0.15, y=0.95):
    ax.text(x, y, label, transform=ax.transAxes,
            ha='right', va='bottom', clip_on=False)

def to_py_datetime(t_np64):
    return np.array(t_np64, dtype='datetime64[ms]').astype('datetime64[ms]').astype(object)

# 軌道データ
t_pos_py = to_py_datetime(THA_SM_pos.time.values)
t_pos_num = mdates.date2num(t_pos_py)               # Matplotlib の日数

R   = np.asarray(THA_rmlatmlt_R, dtype=float)  # Re
mlat= np.asarray(THA_rmlatmlt_MLAT, dtype=float)  # deg
mlt = np.asarray(THA_rmlatmlt_MLT, dtype=float)  # hour [0,24)

# --- MLT の 24h 周期をほどいてから補間し、最後に 24 で折り返す ---
mlt_unwrap = np.unwrap(mlt * 2*np.pi/24.0) * 24.0/(2*np.pi)

# 補間関数（tick の x は「日数」なのでそのまま使う）
def interp_at(x_num):
    Ri    = np.interp(x_num, t_pos_num, R, left=np.nan, right=np.nan)
    mlati = np.interp(x_num, t_pos_num, mlat, left=np.nan, right=np.nan)
    mltiu = np.interp(x_num, t_pos_num, mlt_unwrap, left=np.nan, right=np.nan)
    mlti  = np.mod(mltiu, 24.0)
    return Ri, mlati, mlti

# 目盛フォーマッタ
def rmlt_formatter(x, pos=None):
    Ri, mlati, mlti = interp_at(x)
    if np.any(~np.isfinite([Ri, mlati, mlti])):
        return ""  # 範囲外は空
    return (f"{Ri:0.2f}\n"
            f"{mlati:0.2f}\n"
            f"{mlti:0.2f}")

# セカンダリ x 軸（底 side）を作ってラベルを差し替え
secax = ax_7.secondary_xaxis('bottom', functions=(lambda x: x, lambda x: x))
secax.xaxis.set_major_formatter(mticker.FuncFormatter(rmlt_formatter))

# メインの時間ラベルと重ならないよう余白を広げる
ax_7.tick_params(axis='x', which='major', pad=2)      # 時間ラベル
secax.tick_params(axis='x', which='major', pad=30)  # R/MLAT/MLTラベル

# 好みで：目盛間隔をメイン x と合わせる
secax.set_ticks(ax_7.get_xticks())

fig.text(0.01, 0.100, "hhmm", ha='center', va='center')
fig.text(0.01, 0.083, r"R [$R_{\mathrm{E}}$]", ha='center', va='center')
fig.text(0.01, 0.063, r"MLAT", ha='center', va='center')
fig.text(0.01, 0.045, r"MLT", ha='center', va='center')

add_panel_label(ax_1, '(c-1)')
add_panel_label(ax_2, '(c-2)')
add_panel_label(ax_3, '(c-3)')
add_panel_label(ax_4, '(c-4)')
add_panel_label(ax_5, '(c-5)')
add_panel_label(ax_6, '(c-6)')
add_panel_label(ax_7, '(c-7)')

fig.suptitle('THEMIS-A', y=0.9)

plt.tight_layout()
plt.show()

fig.savefig(r"/mnt/j/KAW_observation/Figure_1_c.png", bbox_inches='tight')

# SYM-H index, AE indexのplot

In [ ]:
psp.omni.data(trange=time_range, no_update=True)
print(pt.tplot_names())

In [ ]:
da_sym_h    = pt.data_quants['SYM_H']
da_ae_index = pt.data_quants['AE_INDEX']
da_au_index = pt.data_quants['AU_INDEX']
da_al_index = pt.data_quants['AL_INDEX']

print(da_sym_h)
print(da_ae_index)
print(da_au_index)
print(da_al_index)

In [ ]:
from datetime import datetime

time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

da_sym_h_analysis       = da_sym_h.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_ae_index_analysis    = da_ae_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_au_index_analysis    = da_au_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_al_index_analysis    = da_al_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(10, 10))

gs      = fig.add_gridspec(3, 1)
ax_0    = fig.add_subplot(gs[0, 0])
ax_1    = fig.add_subplot(gs[1:, 0])

ax_0.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(da_sym_h_analysis.time,       da_sym_h_analysis.data,     lw=1, c='k')
ax_1.plot(da_ae_index_analysis.time,    da_ae_index_analysis.data,  lw=1, c='r',        label='AE')
ax_1.plot(da_au_index_analysis.time,    da_au_index_analysis.data,  lw=1, c='b',        label='AU')
ax_1.plot(da_al_index_analysis.time,    da_al_index_analysis.data,  lw=1, c='green',    label='AL')

ax_0.set_ylabel(r'SYM-H index'          + '\n' + '[nT]')
ax_1.set_ylabel(r'AE, AU, AL indices'   + '\n' + '[nT]')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)

ax_1.legend(ncol=3, fontsize=15)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_1.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)

fig.tight_layout()

plt.show(fig)

In [ ]:
ds_B_time_base  = (ds_B.time[1] - ds_B.time[0]) / np.timedelta64(1, 's')
ds_B_GSM    = ds_B.rolling(time=int(100/ds_B_time_base), center=True).mean()
theta_GSM   = np.rad2deg(np.atan2(ds_B_GSM['B16_gsm_z'], np.sqrt((ds_B_GSM['B16_gsm_x'])**2E0 + (ds_B_GSM['B16_gsm_y'])**2E0)))

In [ ]:
psp.themis.mom(trange=time_range, probe='a', level='l2', no_update=True)
Velocity_ion_gsm    = pt.data_quants['tha_peim_velocity_gsm']   # [km/s]
Velocity_ion_gsm_time_base  = (Velocity_ion_gsm.time[1] - Velocity_ion_gsm.time[0]) / np.timedelta64(1, 's')
Velocity_ion_gsm    = Velocity_ion_gsm.rolling(time=int(100/Velocity_ion_gsm_time_base), center=True).mean()

In [ ]:
from datetime import datetime

mpl.rcParams['font.size'] = 18

time_range_analysis     = ['20220901/22:25:00', '20220901/23:15:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

da_ae_index_analysis    = da_ae_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_au_index_analysis    = da_au_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_al_index_analysis    = da_al_index.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_B_GSM_z_analysis     = ds_B_GSM['B16_gsm_z'].sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
theta_GSM_analysis      = theta_GSM.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
Velocity_ion_gsm_analysis   = Velocity_ion_gsm.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(10, 15))

gs      = fig.add_gridspec(7, 1, hspace=0.05)
ax_0    = fig.add_subplot(gs[0:2, 0])
ax_1    = fig.add_subplot(gs[2, 0], sharex=ax_0)
ax_2    = fig.add_subplot(gs[3, 0], sharex=ax_0)
ax_3    = fig.add_subplot(gs[4, 0], sharex=ax_0)
ax_4    = fig.add_subplot(gs[5, 0], sharex=ax_0)
ax_5    = fig.add_subplot(gs[6, 0], sharex=ax_0)

ax_0.tick_params(axis='x', which='both', labelbottom=False)
ax_1.tick_params(axis='x', which='both', labelbottom=False)
ax_2.tick_params(axis='x', which='both', labelbottom=False)
ax_3.tick_params(axis='x', which='both', labelbottom=False)
ax_4.tick_params(axis='x', which='both', labelbottom=False)

ax_0.plot(da_ae_index_analysis.time,        da_ae_index_analysis.data,              lw=1,   c='r',      label='AE')
ax_0.plot(da_au_index_analysis.time,        da_au_index_analysis.data,              lw=1,   c='b',      label='AU')
ax_0.plot(da_al_index_analysis.time,        da_al_index_analysis.data,              lw=1,   c='green',  label='AL')
ax_1.plot(da_B_GSM_z_analysis.time,         da_B_GSM_z_analysis.data,               lw=1,   c='green')
ax_2.plot(theta_GSM_analysis.time,          theta_GSM_analysis.data,                lw=1,   c='k')
ax_3.plot(Velocity_ion_gsm_analysis.time,   Velocity_ion_gsm_analysis.data[:, 0],   lw=1,   c='r'    )
ax_4.plot(Velocity_ion_gsm_analysis.time,   Velocity_ion_gsm_analysis.data[:, 1],   lw=1,   c='b'    )
ax_5.plot(Velocity_ion_gsm_analysis.time,   Velocity_ion_gsm_analysis.data[:, 2],   lw=1,   c='green')

ax_0.set_ylabel(r'AE, AU, AL indices'       + '\n' + '[nT]')
ax_1.set_ylabel(r'$B_{\mathrm{GSM}z}$'      + '\n' + '[nT]')
ax_2.set_ylabel(r'$\theta_{\mathrm{GSM}}$'  + '\n' + '[deg]')
ax_3.set_ylabel(r'$V_{\mathrm{i}x}$'        + '\n' + '[km/s]')
ax_4.set_ylabel(r'$V_{\mathrm{i}y}$'        + '\n' + '[km/s]')
ax_5.set_ylabel(r'$V_{\mathrm{i}z}$'        + '\n' + '[km/s]')

ax_0.minorticks_on()
ax_0.grid(which='both', alpha=0.5)
ax_1.minorticks_on()
ax_1.grid(which='both', alpha=0.5)
ax_2.minorticks_on()
ax_2.grid(which='both', alpha=0.5)
ax_3.minorticks_on()
ax_3.grid(which='both', alpha=0.5)
ax_4.minorticks_on()
ax_4.grid(which='both', alpha=0.5)
ax_5.minorticks_on()
ax_5.grid(which='both', alpha=0.5)

ax_0.legend(ncol=2, fontsize=15)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_5.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)
ax_5.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))

fig.tight_layout()

plt.show(fig)

# HFAのplot

In [ ]:
psp.erg.pwe_hfa(trange=time_range, level='l2', no_update=True)
psp.erg.pwe_hfa(trange=time_range, level='l3', no_update=True)
psp.erg.mgf(trange=time_range, level='l2', datatype='64hz', no_update=True)

In [ ]:
B_total = np.sqrt(pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].isel(v_dim=0)**2E0 + pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].isel(v_dim=1)**2E0 + pt.data_quants['erg_mgf_l2_mag_64hz_dsi'].isel(v_dim=2)**2E0)
dt = (B_total.time[1] - B_total.time[0]) / np.timedelta64(1, 's')
B_total = B_total.rolling(time=int(100 / dt), center=True).mean('time')

ep_0 = 8.8541878188E-12 #[A^2 s^4 / kg / m^3]
m_e  = 9.1093837E-31    #[kg]
elementary_charge = 1.60217663E-19  #[A s]

n_e = pt.data_quants['erg_pwe_hfa_l3_1min_ne_mgf'].interp(time=B_total.time, method='linear')

f_ce = elementary_charge * B_total*1E-9 / m_e / 2 / np.pi / 1E3                     # [kHz]
f_pe = np.sqrt(n_e*1E6 * elementary_charge**2E0 / m_e / ep_0) / 2 / np.pi / 1E3     # [kHz]

f_UHR = np.sqrt(f_ce**2E0 + f_pe**2E0)  # [kHz]

da_f_UHR    = xr.DataArray(data=f_UHR, dims=['time'], coords={'time': B_total.time}, name='f_UHR')
da_f_ce     = xr.DataArray(data=f_ce,  dims=['time'], coords={'time': B_total.time}, name='f_ce')
da_f_pe     = xr.DataArray(data=f_pe,  dims=['time'], coords={'time': B_total.time}, name='f_pe')

In [ ]:
da_hfa_spectral = pt.data_quants['erg_pwe_hfa_l2_low_spectra_esum']
da_hfa_spectral

In [ ]:
from datetime import datetime
import matplotlib.dates as mdates
from matplotlib.colors import LogNorm


time_range_analysis     = ['20220901/22:25:00', '20220901/23:25:00']
time_range_T_analysis   = [time_range_analysis[0].replace('/', 'T'), time_range_analysis[1].replace('/', 'T')]

da_hfa_spectral_analysis    = da_hfa_spectral.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_f_UHR_analysis           = da_f_UHR.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_f_ce_analysis            = da_f_ce.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))
da_f_pe_analysis            = da_f_pe.sel(time=slice(time_range_T_analysis[0], time_range_T_analysis[1]))

fig = plt.figure(figsize=(12, 5))

ax_0    = fig.add_subplot(111)

T_0 = mdates.date2num(da_hfa_spectral_analysis.time.values)
F_0 = da_hfa_spectral_analysis.spec_bins.values
Z_0 = da_hfa_spectral_analysis.values.astype(float)

T_0_m = np.tile(T_0, (F_0.size, 1)).T
F_0_m = np.tile(F_0, (T_0.size, 1))

pcm = ax_0.pcolormesh(T_0_m, F_0_m, Z_0, shading='auto', norm=LogNorm(vmin=1E-8, vmax=1E-4), cmap='turbo')
ax_0.plot(da_f_UHR_analysis.time, da_f_UHR_analysis.data, c='white', lw=2)
ax_0.minorticks_on()
#ax_0.set_yscale('log')
ax_0.set_ylim(5E0, 3E1)
ax_0.set_ylabel('PWE-HFA [kHz]')
ax_0.grid(which='both', alpha=0.5)

#ax_1.legend(ncol=3, fontsize=15)

to_npdt = lambda s: np.datetime64(datetime.strptime(s, '%Y%m%d/%H:%M:%S'))
time_range_T_analysis_0, time_range_T_analysis_1 = map(to_npdt, time_range_analysis)
ax_0.set_xlim(time_range_T_analysis_0, time_range_T_analysis_1)

cax = ax_0.inset_axes([1.01, 0.05, 0.02, 0.9])
cb = plt.colorbar(pcm, cax=cax)
cb.minorticks_on()
cb.set_label(r'$[\mathrm{(mV/m)}^{2} / \mathrm{Hz}]$')

fig.tight_layout()

plt.show(fig)